# Phase 4–10 · End-to-End Generation Pipeline

**Goal**: Generate the final multi-million row synthetic dataset (accounts + transactions) and all derived features for the detection agents, complete with injected AML patterns and ground-truth labels.

This notebook executes the remaining 7 phases sequentially:
- **Phase 4**: Synthetic Account Generation (using Phase 3 knowledge)
- **Phase 5**: Core Transaction Generation (graph-aware)
- **Phase 6**: AML Pattern Injection (11 fraud scenarios)
- **Phase 7**: Transaction Enrichment (merging account metadata)
- **Phase 8**: Feature Engineering (temporal, velocity, geo)
- **Phase 9**: Constraint Validation (checking data quality)
- **Phase 10**: Dataset Assembly (saving final CSVs and logs)

> **Note on Scale**: Generating 5M rows locally requires significant memory. 
> Change the `N_ACCOUNTS` and `N_TRANSACTIONS` configuration parameters below.

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Cell 1 · Imports & Environment Setup
# ──────────────────────────────────────────────────────────────────────────────
import os, sys, logging, warnings, time
from pathlib import Path
import pandas as pd

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')
logger = logging.getLogger('end_to_end_generation')

# ── Resolve project root ──────────────────────────────────────────────────────
if 'KAGGLE_KERNEL_RUN_TYPE' in os.environ:
    PROJECT_ROOT = Path('/kaggle/working/neural-sentinel')
else:
    PROJECT_ROOT = Path(os.getcwd()).resolve()
    while not (PROJECT_ROOT / 'AGENTS.md').exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
        PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

print(f'Project root: {PROJECT_ROOT}')

# ── Directories ───────────────────────────────────────────────────────────────
INTERIM_DIR   = PROJECT_ROOT / 'data' / 'interim'
GENERATED_DIR = PROJECT_ROOT / 'data' / 'generated'
KB_DIR        = INTERIM_DIR / 'knowledge_base'

GENERATED_DIR.mkdir(parents=True, exist_ok=True)

print('\n✓ Imports complete')

## Configuration (Set target dataset scale)

In [ ]:
# ── Target Data Scale ──
# For local testing, 10k accounts and 100k transactions is good.
# For final Kaggle run, use 50k accounts and 5M transactions.
N_ACCOUNTS = 10_000
N_TRANSACTIONS = 100_000
SEED = 42

# AML Scenario count per scenario (11 scenarios)
# 200 injections per scenario means ~2200 base fraud transactions, plus whatever 
# fan-in/fan-out/layering chains add.
N_INJECTIONS = max(100, N_TRANSACTIONS // 1000)

## Phase 3.5 · Load Knowledge Base

In [ ]:
from src.generation.core.knowledge_extractor import load_knowledge_base

logger.info("Loading knowledge base...")
knowledge = load_knowledge_base(PROJECT_ROOT)
print(f"✓ Knowledge base loaded (contains {len(knowledge)} modules)")

## Phase 4 · Synthetic Account Generation

In [ ]:
%%time
from src.generation.core.account_generator import AccountGenerator

logger.info(f"Generating {N_ACCOUNTS:,} accounts...")
account_gen = AccountGenerator(knowledge, seed=SEED)
synthetic_accounts = account_gen.generate(N_ACCOUNTS)

print(f"\n✓ Generated accounts: {len(synthetic_accounts):,} rows")
print(f"Mule rate: {synthetic_accounts['is_mule'].mean():.2%}")
synthetic_accounts.head(3)

## Phase 5 · Core Transaction Generation

In [ ]:
%%time
from src.generation.core.transaction_generator import TransactionGenerator

logger.info(f"Generating {N_TRANSACTIONS:,} core transactions...")
tx_gen = TransactionGenerator(knowledge, synthetic_accounts, seed=SEED)
core_transactions = tx_gen.generate(N_TRANSACTIONS)

print(f"\n✓ Generated transactions: {len(core_transactions):,} rows")
print(f"Cross border rate: {core_transactions['is_cross_border'].mean():.2%}")
core_transactions.head(3)

## Phase 6 · AML Pattern Injection

In [ ]:
%%time
from src.generation.core.aml_pattern_injector import AMLPatternInjector

logger.info(f"Injecting 11 AML scenarios ({N_INJECTIONS} instances each)...")
injector = AMLPatternInjector(config={"num_injections": N_INJECTIONS, "seed": SEED})

fraud_transactions = injector.inject_all_patterns(core_transactions, synthetic_accounts)

print(f"\n✓ Pattern injection complete.")
print(f"New dataset size: {len(fraud_transactions):,} rows")
print(f"Total fraud rate: {fraud_transactions['is_fraud'].mean():.2%}")
display(fraud_transactions['fraud_type'].value_counts())

## Phase 7 · Transaction Enrichment

In [ ]:
%%time
from src.generation.core.enricher import TransactionEnricher

logger.info("Enriching transactions with account metadata...")
enricher = TransactionEnricher(synthetic_accounts)
enriched_transactions = enricher.enrich(fraud_transactions)

print(f"\n✓ Enrichment complete. Columns: {len(enriched_transactions.columns)}")
enriched_transactions.head(3)

## Phase 8 · Feature Engineering

In [ ]:
%%time
from src.generation.core.feature_engineer import FeatureEngineer

logger.info("Engineering derived ML features (temporal, velocity, geo)...")
engineer = FeatureEngineer()
feature_transactions = engineer.engineer(enriched_transactions)

print(f"\n✓ Feature engineering complete. Columns: {len(feature_transactions.columns)}")
feature_transactions.head(3)

## Phase 9 · Constraint Validation

In [ ]:
%%time
from src.generation.core.validator import ConstraintValidator

logger.info("Validating constraints...")
validator = ConstraintValidator(synthetic_accounts, knowledge=knowledge, drop_invalid=False)

val_accounts, acc_report = validator.validate_accounts()
val_transactions, tx_report = validator.validate_transactions(feature_transactions)

print("\n✓ Validation complete")
print("Account constraints:")
for k, v in acc_report.items(): print(f"  {k}: {v}")
print("\nTransaction constraints:")
for k, v in tx_report.items(): print(f"  {k}: {v}")

## Phase 10 · Dataset Assembly

In [ ]:
%%time
from src.generation.core.dataset_builder import DatasetBuilder

logger.info(f"Building final dataset in {GENERATED_DIR}...")
builder = DatasetBuilder(GENERATED_DIR)

paths = builder.build(
    accounts=val_accounts, 
    transactions=val_transactions,
    schema_report=None,  # We skip schema report here, usually done in Phase 1
    validation_report={"accounts": acc_report, "transactions": tx_report}
)

print("\n✓ Dataset fully assembled and written to disk:")
for k, p in paths.items():
    print(f"  {k}: {p.name}")